# 02 Split

Create training data and Norwegian test/validation users.

- **train**: all ratings from users outside Norway.
- **history**: the first 5 ratings (by date) of each Norwegian user with at least 15 ratings.
- **hidden**: the rest of their ratings. Only used as the answer key, never for training.
- Norwegian users are split 50/50 into validation and test (`SEED = 42`).

In [1]:
import os
from pathlib import Path

# Run from the project folder so `src` can be imported
if Path.cwd().name == "notebooks":
    os.chdir("..")

## Make and save the split

In [2]:
from src.data import load_clean_ratings
from src.split import make_split, save_split

train, history, hidden = make_split(load_clean_ratings())
save_split(train, history, hidden)

## Overview

In [3]:
print(f"train:   {len(train):,} ratings, {train['user_id'].nunique():,} users")
for name, table in [("history", history), ("hidden", hidden)]:
    print(f"\n{name}:")
    print(table.groupby("group").agg(users=("user_id", "nunique"), ratings=("beer_id", "size")))

train:   4,743,275 ratings, 36,106 users

history:
            users  ratings
group                     
test           90      450
validation     90      450

hidden:
            users  ratings
group                     
test           90    86041
validation     90    95984


## Checks

In [4]:
# No Norwegian ratings in train, and every user has exactly 5 history ratings
assert (train["user_country"] != "Norway").all()
assert (history.groupby("user_id").size() == 5).all()

# Hidden ratings come after history for every user
last_history = history.groupby("user_id")["date"].max()
first_hidden = hidden.groupby("user_id")["date"].min()
assert (first_hidden >= last_history[first_hidden.index]).all()

# Each user is in only one group
assert hidden.groupby("user_id")["group"].nunique().max() == 1
print("All checks passed")

All checks passed


Hidden ratings per user (how much each test user has left to find):

In [5]:
hidden.groupby("user_id").size().describe()

count      180.000000
mean      1011.250000
std       1978.253388
min         10.000000
25%         28.000000
50%        145.500000
75%        891.000000
max      10478.000000
dtype: float64